# Step 1: Data Profiling & Quality Audit

**Goal:** understand every table *before* changing anything, and write down each problem in an **issue log**.

Ground rules:
1. Files in `data/raw/` are never modified. Cleaning happens in Step 2.
2. Every table is loaded as **strings** (`dtype=str`). That way pandas can't quietly convert, round or fill values and hide problems from you.
3. Every problem you find goes into `log_issue(...)`. The log becomes `reports/data_quality_log.csv`, and in the interview it backs up the line "here's everything I found and how I fixed it."

For each section, read the output and answer the **❓ questions** in your own words.

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
REPORTS = Path("../reports"); REPORTS.mkdir(exist_ok=True)

users   = pd.read_csv(RAW / "users.csv", dtype=str)
events  = pd.read_csv(RAW / "events.csv", dtype=str)
assess  = pd.read_csv(RAW / "assessments.csv", dtype=str)
assign  = pd.read_csv(RAW / "experiment_assignments.csv", dtype=str)
content = pd.read_csv(RAW / "content.csv", dtype=str)
mkt     = pd.read_csv(RAW / "marketing_spend.csv", dtype=str)

for name, df in [("users", users), ("events", events), ("assessments", assess),
                 ("assignments", assign), ("content", content), ("marketing", mkt)]:
    print(f"{name:12s} {df.shape}")

## Helpers: column profiler and issue log

In [ ]:
def profile(df, max_examples=5):
    """One row per column: nulls, cardinality, and example values."""
    rows = []
    for c in df.columns:
        s = df[c]
        rows.append({
            "column": c,
            "n_null": s.isna().sum(),
            "pct_null": round(100 * s.isna().mean(), 2),
            "n_unique": s.nunique(dropna=True),
            "examples": list(s.dropna().unique()[:max_examples]),
        })
    return pd.DataFrame(rows)

def shape_pattern(s):
    """Replace digits with 'd' and letters with 'a' to reveal format families in a string column."""
    return s.fillna("<NULL>").str.replace(r"[A-Za-z]", "a", regex=True).str.replace(r"\d", "d", regex=True)

ISSUES = []
def log_issue(table, column, issue, evidence, severity, proposed_fix):
    """severity: high (changes conclusions) / medium / low"""
    ISSUES.append(dict(table=table, column=column, issue=issue, evidence=evidence,
                       severity=severity, proposed_fix=proposed_fix))

# Example usage (delete or edit once you've confirmed it yourself):
# log_issue("users", "user_id", "Duplicate user rows", "140 duplicated ids", "high", "Dedupe, keep legacy row")

---
## 1. `users`
❓ Is `user_id` really unique? If it isn't, how do the duplicates differ from the originals?
❓ Do categorical columns use one consistent set of labels?
❓ Which columns are missing data, and is that missingness random or structural (missing for a reason)?

In [ ]:
profile(users)

In [ ]:
print("rows:", len(users), "| unique user_id:", users.user_id.nunique())
dup_ids = users.loc[users.user_id.duplicated(keep=False)].sort_values("user_id")
dup_ids.head(10)

In [ ]:
for c in ["platform", "signup_app_version", "country", "age_band", "gender",
          "acquisition_channel", "meditation_experience", "primary_goal", "source_system"]:
    print(f"\n--- {c}"); print(users[c].value_counts(dropna=False).to_string())

In [ ]:
# ID formats and timestamp formats
print(shape_pattern(users.user_id).value_counts().head(), "\n")
print(shape_pattern(users.signup_ts).value_counts().head())

In [ ]:
# Is missingness related to anything else? Example: onboarding answers vs source system / platform
users.assign(exp_missing=users.meditation_experience.isna()) \
     .groupby(["source_system", "platform"]).exp_missing.mean().round(3).unstack()

---
## 2. `events` (most of the problems will be here)
❓ How many timestamp *formats* are there? Which rows have each one?
❓ Do the event names match the README's tracking plan? Which names don't, and when do they show up?
❓ Do the `properties` keys stay the same within an event type?
❓ Are there duplicates? Check both exact copies and near-copies a second or two apart.
❓ Are there users whose activity is humanly impossible?
❓ Are there dates or volumes that look wrong when you plot them over time?

In [ ]:
profile(events)

In [ ]:
print(events.event_name.value_counts().to_string(), "\n")
print(events.platform.value_counts().to_string())

In [ ]:
# Timestamp format families
events["ts_pattern"] = shape_pattern(events.event_ts)
events.ts_pattern.value_counts()

In [ ]:
# Which users / platforms produce each timestamp format?
pd.crosstab(events.ts_pattern, events.platform)

In [ ]:
# Parse timestamps (format="mixed" for now; you decide the correct handling in Step 2)
events["ts_parsed"] = pd.to_datetime(events.event_ts, format="mixed", utc=True, errors="coerce")
print("unparseable:", events.ts_parsed.isna().sum())
print("min:", events.ts_parsed.min(), "| max:", events.ts_parsed.max())
events.ts_parsed.dt.year.value_counts().sort_index()

In [ ]:
# Event names x schema version x app version
pd.crosstab([events.event_name], [events.schema_version])

In [ ]:
# Property keys per event type & schema version
events["prop_keys"] = events.properties.map(lambda p: ",".join(sorted(json.loads(p).keys())) if isinstance(p, str) else "<NULL>")
events.groupby(["event_name", "schema_version"]).prop_keys.value_counts().to_frame("n")

In [ ]:
# Duration values: units sanity check (look at the magnitudes)
def get_duration(p):
    d = json.loads(p)
    return d.get("duration_sec", d.get("durationMs"))
comp = events[events.event_name.isin(["practice_complete", "meditation_completed"])].copy()
comp["duration_raw"] = comp.properties.map(get_duration).astype(float)
comp.groupby("schema_version").duration_raw.describe()

In [ ]:
# Exact duplicates (ignoring event_id)
key_cols = ["user_id", "event_name", "event_ts", "session_id", "properties"]
print("exact dupes:", events.duplicated(subset=key_cols).sum())

# Near-duplicates: same user/event/session/properties within 3 seconds
e = events.dropna(subset=["ts_parsed"]).sort_values(["user_id", "event_name", "session_id", "properties", "ts_parsed"])
gap = e.groupby(["user_id", "event_name", "session_id", "properties"]).ts_parsed.diff().dt.total_seconds()
near = e[gap.between(0, 3)]
print("near-dupes (<=3s):", len(near))
near.platform.value_counts()

In [ ]:
# Null user_id events
print(events.user_id.isna().sum())
events[events.user_id.isna()].event_name.value_counts()

In [ ]:
# Events per user per day: who is superhuman?
e = events.dropna(subset=["user_id", "ts_parsed"])
per_day = e.groupby([e.user_id, e.ts_parsed.dt.date]).size()
print(per_day.describe(percentiles=[.5, .9, .99, .999]))
top = per_day.groupby(level=0).max().sort_values(ascending=False).head(10)
users.set_index("user_id").loc[top.index.intersection(users.user_id)]

In [ ]:
# Daily volume of key events by platform: look for drops, spikes, gaps
e = events.dropna(subset=["ts_parsed"])
e = e[(e.ts_parsed >= "2026-06-01") & (e.ts_parsed <= "2026-11-16")]
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, names in zip(axes, [["practice_start", "meditation_started"], ["practice_complete", "meditation_completed"]]):
    sub = e[e.event_name.isin(names)]
    sub.groupby([sub.ts_parsed.dt.date, "platform"]).size().unstack().plot(ax=ax)
    ax.set_title(" / ".join(names)); ax.set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
# Orphans: events from users not in the users table
known = set(users.user_id)
orphan = events.user_id.notna() & ~events.user_id.isin(known)
print("orphan events:", orphan.sum())

---
## 3. `assessments`
❓ Do the `wave` labels match the README?
❓ Are all item values within the valid range? How often does the `99` code appear?
❓ Does any user have more than one submission for the same wave?
❓ Are there low-effort responses (very fast, or the same answer to every item)?
❓ **Psychometrics:** does every item correlate positively with the rest of its pillar once you apply the README's scoring rules? Try splitting the result by other columns.

In [ ]:
profile(assess)

In [ ]:
print(pd.crosstab(assess.wave, assess.scoring_version), "\n")
items = [f"q{i:02d}" for i in range(1, 17)]
phq = [f"phq4_{i}" for i in range(1, 5)]
pd.concat({c: assess[c].value_counts() for c in items + phq}, axis=1).fillna(0).astype(int)

In [ ]:
dup = assess.groupby(["user_id", "wave"]).size()
print(dup.value_counts())

In [ ]:
A = assess.copy()
A[items + phq] = A[items + phq].astype(int).replace(99, np.nan)
A["completion_seconds"] = A.completion_seconds.astype(int)
A["item_sd"] = A[items].std(axis=1)
print(A.completion_seconds.describe(percentiles=[.01, .05, .5, .95]))
print("zero-variance (straight-line) responses:", (A.item_sd == 0).sum())

In [ ]:
# Item-rest correlation, scored per the README (reverse q04, q08, q12, q16)
def item_rest(df):
    X = df[items].where((df[items] >= 1) & (df[items] <= 5))
    for q in ["q04", "q08", "q12", "q16"]:
        X[q] = 6 - X[q]
    out = {}
    for p, block in enumerate([items[i:i+4] for i in range(0, 16, 4)]):
        for q in block:
            rest = X[[b for b in block if b != q]].mean(axis=1)
            out[q] = X[q].corr(rest)
    return pd.Series(out).round(2)

item_rest(A).to_frame("all_rows")
# ❓ Now try: item_rest on subsets of A (e.g., by a version or system column). Anything change?

---
## 4. `experiment_assignments`
❓ Is each user assigned exactly once? Is anyone in **both** variants?
❓ Is the split really 50/50? Run a **Sample Ratio Mismatch (SRM)** test.
❓ If there's a mismatch, where does it come from? Think about how to get every user who *should* have been assigned, using the eligibility rule in the README.

In [ ]:
print("rows:", len(assign), "| unique users:", assign.user_id.nunique())
print("exact duplicate rows:", assign.duplicated(subset=["user_id", "variant", "assigned_ts"]).sum())
nvar = assign.groupby("user_id").variant.nunique()
print("users in both variants:", (nvar > 1).sum())

In [ ]:
# SRM check on first assignment per user
first = assign.sort_values("assigned_ts").drop_duplicates("user_id")
counts = first.variant.value_counts()
chi = stats.chisquare(counts.values)
print(counts.to_string()); print(f"SRM chi-square p = {chi.pvalue:.2e}")

In [ ]:
# Coverage: eligible users (per README rule) vs logged assignments
u = users.drop_duplicates("user_id").copy()
u["signup"] = pd.to_datetime(u.signup_ts, utc=True)
eligible = u[(u.signup >= "2026-09-01") & (u.signup < "2026-10-01") & u.platform.str.lower().isin(["ios", "android"])]
eligible = eligible.assign(assigned=eligible.user_id.isin(set(assign.user_id)))
print("eligible:", len(eligible), "| assigned:", eligible.assigned.sum())
eligible.groupby(["platform", "signup_app_version"]).assigned.agg(["size", "mean"]).round(3)

---
## 5. `content` & `marketing_spend`
❓ Does every `content_id` that appears in events exist in the catalog?
❓ How do the platform-reported installs compare with the signups in `users`?

In [ ]:
print(profile(content))
ids_in_events = set(events.properties.map(lambda p: json.loads(p).get("content_id") or json.loads(p).get("contentId")).dropna())
print("content ids in events not in catalog:", len(ids_in_events - set(content.content_id)))

In [ ]:
m = mkt.astype({"spend_usd": float, "attributed_installs": int})
paid = u[u.acquisition_channel.isin(["paid_social", "app_store_search"])]
first_party = paid.groupby("acquisition_channel").size()
summary = m.groupby("channel")[["spend_usd", "attributed_installs"]].sum().join(first_party.rename("signups_in_users"))
summary["cac_platform"] = summary.spend_usd / summary.attributed_installs
summary["cac_first_party"] = summary.spend_usd / summary.signups_in_users
summary.round(2)

---
## 6. Wrap-up: save the issue log
Before running this, make sure you've added a `log_issue(...)` call for every problem you found above.

In [ ]:
log = pd.DataFrame(ISSUES)
log.to_csv(REPORTS / "data_quality_log.csv", index=False)
log